# Case 0001: GT vs Model GUI

This notebook provides an interactive GUI to compare GT and model prediction for case 0001.

Data source:
- `pred_npz/case_0001_predictions.npz`

Controls:
- `Timestep`
- `Field` (`Bx`, `By`, `|B|`)
- `Mode` (`scatter`, `tri`)
- `Mesh` overlay toggle

In [1]:
%matplotlib inline
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    import h5py
except ImportError:
    h5py = None

BASE = Path('.').resolve()
NPZ_PATH = BASE / 'pred_npz' / 'case_0001_predictions.npz'
if not NPZ_PATH.exists():
    raise FileNotFoundError(f'Prediction NPZ not found: {NPZ_PATH}')

arr = np.load(NPZ_PATH)
pos_all = arr['pos']
true_bxy_all = arr['true_bxy']
pred_bxy_all = arr['pred_bxy']
steps = arr['steps']
times = arr['times']
case_index = int(arr['case_index']) if np.ndim(arr['case_index']) == 0 else int(arr['case_index'][0])

n_steps = int(pos_all.shape[0])
n_nodes = int(pos_all.shape[1])
print(f'Loaded: {NPZ_PATH}')
print(f'case={case_index}, n_steps={n_steps}, n_nodes={n_nodes}')

mesh_triangles = None
if h5py is not None:
    h5_file = BASE / 'doe_data' / 'case_0001' / 'postproc' / 'Mag_OnLoadTorque_result_1.h5'
    if h5_file.exists():
        try:
            with h5py.File(h5_file, 'r') as f:
                n_id = np.asarray(f['mesh/node_id'][:], dtype=np.int32)
                n_1 = np.asarray(f['mesh/node_1'][:], dtype=np.int32)
                n_2 = np.asarray(f['mesh/node_2'][:], dtype=np.int32)
                n_3 = np.asarray(f['mesh/node_3'][:], dtype=np.int32)

                max_nid = int(n_id.max()) + 1
                lut = np.full(max_nid, -1, dtype=np.int64)
                for i, nid in enumerate(np.sort(n_id)):
                    lut[nid] = i

                m = min(len(n_1), len(n_2), len(n_3))
                i1 = lut[np.clip(n_1[:m], 0, max_nid - 1)]
                i2 = lut[np.clip(n_2[:m], 0, max_nid - 1)]
                i3 = lut[np.clip(n_3[:m], 0, max_nid - 1)]
                valid = (i1 >= 0) & (i2 >= 0) & (i3 >= 0)
                mesh_triangles = np.column_stack([i1[valid], i2[valid], i3[valid]])
                print(f'Loaded mesh triangles: {len(mesh_triangles)}')
        except Exception as e:
            print(f'Mesh load failed: {e}')

def get_tri(x, y):
    if mesh_triangles is not None:
        return mtri.Triangulation(x, y, triangles=mesh_triangles)
    return mtri.Triangulation(x, y)

def field_values(step_idx, field_name):
    gt_bx = true_bxy_all[step_idx, :, 0]
    gt_by = true_bxy_all[step_idx, :, 1]
    pd_bx = pred_bxy_all[step_idx, :, 0]
    pd_by = pred_bxy_all[step_idx, :, 1]

    if field_name == 'Bx':
        return gt_bx, pd_bx
    if field_name == 'By':
        return gt_by, pd_by

    gt_mag = np.sqrt(gt_bx**2 + gt_by**2)
    pd_mag = np.sqrt(pd_bx**2 + pd_by**2)
    return gt_mag, pd_mag

Loaded: D:\KDH\NvidiaNemo\pred_npz\case_0001_predictions.npz
case=1, n_steps=45, n_nodes=7013
Loaded mesh triangles: 13448


In [2]:
step_slider = widgets.IntSlider(
    value=min(20, n_steps - 1), min=0, max=n_steps - 1, step=1,
    description='Timestep:', style={'description_width': '70px'},
    layout=widgets.Layout(width='360px')
)
field_dropdown = widgets.Dropdown(
    options=['Bx', 'By', '|B|'], value='Bx',
    description='Field:', style={'description_width': '70px'}
)
mode_dropdown = widgets.Dropdown(
    options=['scatter', 'tri'], value='scatter',
    description='Mode:', style={'description_width': '70px'}
)
show_mesh = widgets.Checkbox(value=False, description='Mesh overlay', indent=False)
info = widgets.HTML(value='')
out = widgets.Output(layout=widgets.Layout(width='100%', overflow='auto'))

def render(*_):
    si = step_slider.value
    field = field_dropdown.value
    mode = mode_dropdown.value

    x = pos_all[si, :, 0]
    y = pos_all[si, :, 1]
    gt_f, pd_f = field_values(si, field)
    err = np.abs(pd_f - gt_f)

    finite = np.isfinite(gt_f) & np.isfinite(pd_f)
    rmse = float(np.sqrt(np.nanmean((pd_f - gt_f)**2)))
    mae = float(np.nanmean(np.abs(pd_f - gt_f)))

    if np.any(finite):
        ss_res = np.sum((pd_f[finite] - gt_f[finite])**2)
        ss_tot = np.sum((gt_f[finite] - gt_f[finite].mean())**2)
        r2 = float(1.0 - ss_res / max(ss_tot, 1e-12))
    else:
        r2 = float('nan')

    info.value = (
        f'<b>Case 0001</b> | step={si} | t={times[si]*1000:.4f} ms | rot={steps[si]} | '
        f'RMSE={rmse:.6f} | MAE={mae:.6f} | R2={r2:.6f}'
    )

    with out:
        clear_output(wait=True)
        fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=110, constrained_layout=True)

        tri = get_tri(x, y)
        vmin = float(np.nanmin([gt_f.min(), pd_f.min()]))
        vmax = float(np.nanmax([gt_f.max(), pd_f.max()]))

        cmap_field = 'RdBu_r' if field != '|B|' else 'jet'

        if mode == 'scatter':
            m0 = axes[0, 0].scatter(x, y, c=gt_f, s=2.0, cmap=cmap_field, vmin=vmin, vmax=vmax, marker='.')
            m1 = axes[0, 1].scatter(x, y, c=pd_f, s=2.0, cmap=cmap_field, vmin=vmin, vmax=vmax, marker='.')
            m2 = axes[1, 0].scatter(x, y, c=err, s=2.0, cmap='hot_r', marker='.')
        else:
            m0 = axes[0, 0].tripcolor(tri, gt_f, shading='flat', cmap=cmap_field, vmin=vmin, vmax=vmax)
            m1 = axes[0, 1].tripcolor(tri, pd_f, shading='flat', cmap=cmap_field, vmin=vmin, vmax=vmax)
            m2 = axes[1, 0].tripcolor(tri, err, shading='flat', cmap='hot_r')

        if show_mesh.value:
            for ax in [axes[0, 0], axes[0, 1], axes[1, 0]]:
                ax.triplot(tri, color='k', lw=0.15, alpha=0.25)

        axes[0, 0].set_title(f'GT {field}')
        axes[0, 1].set_title(f'Prediction {field}')
        axes[1, 0].set_title(f'Abs Error {field}')

        for ax in [axes[0, 0], axes[0, 1], axes[1, 0]]:
            ax.set_aspect('equal')
            ax.set_xlabel('X')
            ax.set_ylabel('Y')

        gt_flat = gt_f.ravel()
        pd_flat = pd_f.ravel()
        ok = np.isfinite(gt_flat) & np.isfinite(pd_flat)
        axes[1, 1].scatter(gt_flat[ok], pd_flat[ok], s=2.0, alpha=0.3, color='#1f77b4')
        if np.any(ok):
            lo = float(min(gt_flat[ok].min(), pd_flat[ok].min()))
            hi = float(max(gt_flat[ok].max(), pd_flat[ok].max()))
            axes[1, 1].plot([lo, hi], [lo, hi], 'k--', linewidth=1.0)
            axes[1, 1].set_xlim(lo, hi)
            axes[1, 1].set_ylim(lo, hi)
        axes[1, 1].set_title(f'Scatter (GT vs Pred), R2={r2:.4f}')
        axes[1, 1].set_xlabel(f'GT {field} [T]')
        axes[1, 1].set_ylabel(f'Pred {field} [T]')
        axes[1, 1].grid(True, alpha=0.3)

        cbar0 = fig.colorbar(m0, ax=[axes[0, 0], axes[0, 1]], orientation='horizontal', fraction=0.05, pad=0.08)
        cbar0.set_label(f'{field} [T]')
        cbar1 = fig.colorbar(m2, ax=[axes[1, 0]], orientation='horizontal', fraction=0.08, pad=0.12)
        cbar1.set_label(f'Abs Error {field} [T]')

        fig.suptitle(f'Case 0001 GT vs Model ({field}, mode={mode})', fontsize=15)
        plt.show()

for w in [step_slider, field_dropdown, mode_dropdown, show_mesh]:
    w.observe(render, names='value')

display(widgets.HBox([step_slider, field_dropdown, mode_dropdown, show_mesh]))
display(info)
display(out)
render()

HTML(value='')

Output(layout=Layout(overflow='auto', width='100%'))